# Asistent de Triaj Medical și Literatură Clinică - Demonstrație Pipeline
Acest notebook prezintă execuția cap-la-cap (end-to-end) a sistemului multi-agent de triaj medical, demonstrând funcționalitatea fiecărui agent și orchestrarea realizată cu LangGraph.

### 1. Configurare Mediu și Încărcare Variabile
Vom încărca cheile API din fișierul `.env` și vom configura importurile necesare.

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Adăugare director rădăcină în path
sys.path.append(os.path.abspath(".."))

load_dotenv("../.env")
print("OpenAI API Key încărcată:", "OPENAI_API_KEY" in os.environ)

### 2. Rularea Pipeline-ului LangGraph pe un caz clinic complex
Vom defini o prezentare clinică complexă (suspiciune de Sepsis) și o vom introduce în graful de stări.

In [ ]:
from src.graph.workflow import compile_and_draw_graph

# Rulăm graful pe o prezentare de sepsis
raw_text = "Femeie de 58 de ani, febră 39.1 grade Celsius, puls 102 bpm, TA 90/60 mmHg, frecvență respiratorie 24, SpO2 93% pe aer ambiental. Prezintă confuzie acută, tuse productivă cu spută purulentă de 3 zile. Fără alergii cunoscute."

# Compilăm graful
app = compile_and_draw_graph()

# Rulăm fluxul
inputs = {
    "raw_clinical_text": raw_text,
    "custom_threshold": 0.75
}

result = app.invoke(inputs)
print("Execuție completă! ID Caz:", result["clinical_case"].case_id)

### 3. Vizualizarea Diagramelor de Sistem
Vom afișa diagramele salvate în timpul execuției: fluxul grafului, heatmap-ul de similaritate semantică și distribuția diagnosticelor.

In [ ]:
from IPython.display import Image, display

print("--- Arhitectura Grafului LangGraph ---")
if os.path.exists("../logs/workflow_graph.png"):
    display(Image(filename="../logs/workflow_graph.png"))
else:
    print("Png-ul grafului nu a putut fi generat (lipsă graphviz/pygraphviz). Puteți vedea codul Mermaid în logs/workflow_graph.txt")

print("\n--- Heatmap Similaritate RAG ---")
if os.path.exists("../logs/retrieval_heatmap.png"):
    display(Image(filename="../logs/retrieval_heatmap.png"))
else:
    print("Heatmap-ul nu este generat încă. Rulați scripts/test_retrieval.py")

### 4. Afișarea Raportului Medical Final Generat
Vom încărca fișierul raportului Markdown generat de nodul final și îl vom reda în format Markdown.

In [ ]:
from IPython.display import Markdown

report_path = result.get("report_path")
if report_path and os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        report_content = f.read()
    display(Markdown(report_content))
else:
    print("Raportul nu a putut fi găsit.")

### 5. Audit Tehnic: Timpi de Execuție și Costuri Estimative
Analizăm timpii consumați de fiecare nod și calculăm costul în USD pe baza numărului estimat de tokeni.

In [ ]:
import pandas as pd

log_runs = result.get("log_runs", [])
df = pd.DataFrame(log_runs)

# Calculăm costul estimativ în USD (Prețuri medii: gpt-4o input $5/M, output $15/M; gpt-4o-mini input $0.15/M, output $0.60/M)
def estimate_cost(row):
    nod = row['nod']
    tokens = row.get('tokens_consumed', 0)
    if nod == "intake_symptoms":
        return (tokens / 1000000) * 0.15
    elif nod == "generate_differential":
        return (tokens / 1000000) * 5.0
    elif nod == "generate_explanation":
        return (tokens / 1000000) * 0.15
    return 0.0

df['estimated_cost_usd'] = df.apply(estimate_cost, axis=1)
display(df)
print(f"\nCost Total Estimat: ${df['estimated_cost_usd'].sum():.6f} USD")
print(f"Timp Total Execuție: {df['duration_seconds'].sum():.2f} secunde")

### 6. Analiza RAGAS și Analiza Halucinațiilor
Citim scorurile RAGAS calculate în logs și comentăm valoarea lor clinică.

In [ ]:
import json

eval_path = "../logs/rag_evaluation.json"
if os.path.exists(eval_path):
    with open(eval_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    print("--- Metricile RAGAS Global ---")
    print(json.dumps(eval_data.get("global_metrics"), indent=2))
else:
    print("Fișierul de evaluare RAGAS nu a fost găsit. Rulați scripts/evaluate_rag.py")

#### Interpretare Clinică Scoruri RAGAS:
- **Faithfulness (Fidelitate) >= 0.85**: Indică faptul că toate afirmațiile din diagnosticul diferențial se bazează direct pe contextul extras (risc minim de halucinație clinică).
- **Answer Relevancy (Relevanța Răspunsului) >= 0.80**: Confirmă că diagnosticele diferențiale răspund direct la acuzele și simptomele pacientului, eliminând elementele redundante.
- **Context Recall (Acoperirea Contextului) >= 0.90**: Demonstrează că agentul de RAG a reușit să extragă majoritatea ghidurilor relevante pentru criteriile diagnostice din baza de date.